In [1]:
using Random
using LinearAlgebra
using JuMP
using MosekTools
using DynamicPolynomials
using SumOfSquares

using ppt2

In [4]:
n = m = 4
rng = Xoshiro(0)
f, h = nothing, nothing

for attempt in 1:20
    for construction in 1:20
        f, h = ppt2.pncp_algorithm(n, m; rng=rng)

        sos, _ = ppt2.solve_sos(n, m, f, h)
        if !sos
            println("construction $construction")
            break
        end
    end

    pos, _ = ppt2.solve_sos(n, m, f, h, 1)
    if pos
        println("attempt $attempt")
        break
    end
end

construction 1
construction 1
attempt 2


In [5]:
model = SOSModel(Mosek.Optimizer)
set_silent(model)

@polyvar X[1:n] Y[1:m]
@variable(model, delta)

XY = kron(X, Y)

f_poly = f' * kron(XY, XY)
h_poly = h' * XY

p = delta * f_poly + (h_poly' * h_poly)

relax = (XY' * XY)
con = @constraint(model, p * relax in SOSCone())
@objective(model, Max, delta)

optimize!(model)

is_solved_and_feasible(model) && value(delta) > 1e-4

true

In [6]:
gram = gram_matrix(con)
Q = Symmetric(Matrix(gram.Q))
b = gram.basis.monomials

vals, vecs = eigen(Q)

vals[1:(n-1)*(m-1)+1] .= 0

G = vecs * Diagonal(vals) * vecs'

G_poly = b' * G * b

basis_p = monomials(p)
basis_G = monomials(G_poly)

coeff_G = DynamicPolynomials.coefficients(G_poly)
A = zeros(BigInt, length(basis_G), length(basis_p))

for i in eachindex(basis_G), j in eachindex(basis_p)
    A[i, j] = DynamicPolynomials.coefficient(basis_p[j] * relax, basis_G[i])
end

coeff_p = rationalize.(A \ coeff_G, tol=1e-10)
p_hat = coeff_p' * basis_p;

In [7]:
test = SOSModel(Mosek.Optimizer)
set_silent(test)

@constraint(test, p_hat in SOSCone())
@objective(test, Min, 0.0)
optimize!(test)

is_solved_and_feasible(test)

false

In [12]:
show(stdout, "text/plain", coeff_p)

100-element Vector{Rational{Int64}}:
    23181//81451
   -30731//125091
    49901//70132
   -51118//238273
   -37851//152354
    11937//71512
   -15194//133757
   -54749//87276
     9073//62265
    11197//53908
    28541//90590
   -35147//110354
   -89527//104480
   102782//122441
  -147459//89536
    14752//84907
   -65980//85401
   214250//88641
    45742//112655
  -122467//182738
   121944//171389
    25165//81444
    84577//112293
   228609//382351
    94815//137386
   178299//168533
  -181389//124825
  -250371//129716
  -155259//63185
   160863//76534
   -24119//22800
   105313//51959
   139020//115531
   -11498//31577
   -37412//93131
     2229//122972
    18426//66875
  -343279//125384
    71055//170326
    30581//43232
 -5085652//2113823
  -300925//128682
  -212157//183017
  -211226//94639
   -83309//122299
   -62569//195516
   340698//53665
   357697//90784
    85969//34322
  -354899//78477
    48143//16568
   252864//108313
    69997//113414
   -71470//126887
   -51729//12631